<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/02_Intermediate/L22_Domain_Adaptation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L22: Domain Adaptation - Adapting Models to Your Domain

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Intermediate  
**Lesson:** 22 of 30

---

## Learning Objectives

By the end of this lesson, you will:
- Adapt a pre-trained model to a specific domain (e.g. medical, legal, code)
- Use continued pre-training (MLM) on in-domain text
- Fine-tune on a downstream task after domain adaptation

---

## Concept: Domain Adaptation

**Domain adaptation** improves performance on in-domain data by: (1) **Continued pre-training** (MLM or CLM) on domain corpora; (2) **Task fine-tuning** on labeled in-domain data; (3) **Vocabulary extension** if the domain has many new terms. We illustrate (1) and (2) with a small in-domain corpus.

---

In [ ]:
!pip install transformers torch datasets -q
from transformers import AutoTokenizer, AutoModelForMaskedLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: In-Domain Corpus (Example)

Use a small list of domain sentences. In practice, use a large corpus (e.g. medical abstracts, legal text).

---

In [ ]:
domain_texts = [
    "The patient presented with fever and cough. Diagnosis was confirmed by PCR.",
    "Treatment includes rest and fluids. Antibiotics are not recommended for viral infections.",
    "The court held that the defendant was liable. The appeal was dismissed.",
    "Code review found a null pointer exception. The fix was merged to main branch.",
] * 20

dataset = Dataset.from_dict({"text": domain_texts})
print(f"Domain corpus: {len(dataset)} lines")

## Part 2: Continued Pre-training (MLM) on Domain

Load a masked LM (e.g. DistilBERT), tokenize the corpus, and run a short MLM training.

---

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

dataset = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)

args = TrainingArguments(
    output_dir="./out_domain_adapt_l22",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=collator)
trainer.train()
print("Continued pre-training done. Save model and use for downstream task.")

## Part 3: Downstream Task After Adaptation

Use the adapted model as the base for sequence classification (or NER, QA): load from the saved checkpoint and fine-tune on your labeled task data.

---

In [ ]:
# In practice: load adapted MLM, add classification head, fine-tune on labeled data.
print("Pipeline: domain MLM -> save -> load as base for AutoModelForSequenceClassification -> fine-tune on task.")

## Exercises

1. Use a larger in-domain corpus (e.g. from HuggingFace) and run 2–3 epochs of MLM.
2. Compare eval accuracy: base model vs domain-adapted model on an in-domain classification task.
3. Add domain-specific tokens to the tokenizer and retrain (optional).

---

## Key Takeaways

1. Domain adaptation = continued pre-training on domain text + task fine-tuning.
2. MLM on in-domain sentences helps the model learn domain vocabulary and style.
3. Save the adapted encoder and reuse for classification/QA/NER.

---

## Next Lesson

**L23: Multi-Task Learning** – Training on multiple tasks with shared representations.

---